In [1]:
import sys
import pathlib

# set pythonpath to the main module directory
module_dir = pathlib.Path("..").parent.resolve().parent
if str(module_dir) not in sys.path:
    sys.path.append(str(module_dir))

In [2]:
import matplotlib.pyplot as plt
import seaborn as sns

# Global seaborn / matplotlib defaults
sns.set_theme(
    style="whitegrid",          # axes with grid
    # palette="colorblind",            # default colour palette
    # font_scale=1.3,             # scales all font sizes uniformly
    rc={
        # "lines.linewidth": 4,           # default line width
        # "axes.titlesize": 16,
        # "axes.labelsize": 22,
        # "axes.labelweight": "bold",     
        # "xtick.labelsize": 18,
        # "ytick.labelsize": 18,
        # "legend.fontsize": 19,
        "grid.linestyle": "-",
        "grid.alpha": 0.6,
        # "lines.markersize": 8,

    },
)

Load data

In [3]:
import pandas as pd


def preview_results(df: pd.DataFrame, sample_size: int = 10) -> None:
    if len(df) > 0:
        display(df.sample(min(sample_size, len(df))))


logprob_results_path = "../analysis/results/logprob_acc_merged.json"
logprob_results = pd.read_json(logprob_results_path, orient="records")

logprob_norm_results_path = "../analysis/results/logprob_acc_norm_merged.json"
logprob_norm_results = pd.read_json(logprob_norm_results_path, orient="records")

generative_results_path = "../analysis/generative_tails.json"
generative_results = pd.read_json(generative_results_path, orient="records")

In [4]:
raw_results_path = "../logs/silent-norm-final-v1/results.json"
raw_results = pd.read_json(raw_results_path, orient="records")

In [5]:
def add_path_metadata(dirty_df: pd.DataFrame) -> pd.DataFrame:
    dirty_df = dirty_df.copy()
    dirty_parts = dirty_df["path"].str.split("/")
    if (dirty_parts.str.len() < 2).any():
        raise ValueError("Clean path does not contain enough segments to parse model_name")

    dirty_out = dirty_df.copy()

    dirty_out["model_name"] = dirty_parts.str[-5]
    dirty_out["train_dataset"] = dirty_parts.str[-4]
    dirty_out["layer_name"] = dirty_parts.str[-3]
    dirty_out["exp_name"] = dirty_parts.str[-2]

    return dirty_out


# apply formatting to metric column
def format_metric_column(df: pd.DataFrame, metric_col: str = "metric") -> pd.DataFrame:
    df = df.copy()
    df[metric_col] = df[metric_col].apply(lambda x: x.replace(",none", ""))
    return df

raw_results = add_path_metadata(raw_results)
raw_results = format_metric_column(raw_results)
raw_results

,path,file,benchmark,metric,value,model_name,train_dataset,layer_name,exp_name
0,/home/fre.gilad/source/silent_direction/logs/s...,anli.json,anli_r1,acc,0.349000,Llama-2-7b-chat-hf,oasst2_tulu-v3,model.embed_tokens,Llama-2-7b-chat-hf-KL-0.0-iter1
1,/home/fre.gilad/source/silent_direction/logs/s...,anli.json,anli_r2,acc,0.325000,Llama-2-7b-chat-hf,oasst2_tulu-v3,model.embed_tokens,Llama-2-7b-chat-hf-KL-0.0-iter1
2,/home/fre.gilad/source/silent_direction/logs/s...,anli.json,anli_r3,acc,0.359167,Llama-2-7b-chat-hf,oasst2_tulu-v3,model.embed_tokens,Llama-2-7b-chat-hf-KL-0.0-iter1
3,/home/fre.gilad/source/silent_direction/logs/s...,blimp.json,blimp,acc,0.763642,Llama-2-7b-chat-hf,oasst2_tulu-v3,model.embed_tokens,Llama-2-7b-chat-hf-KL-0.0-iter1
4,/home/fre.gilad/source/silent_direction/logs/s...,blimp.json,blimp_adjunct_island,acc,0.836000,Llama-2-7b-chat-hf,oasst2_tulu-v3,model.embed_tokens,Llama-2-7b-chat-hf-KL-0.0-iter1
...,...,...,...,...,...,...,...,...,...
27106,/home/fre.gilad/source/silent_direction/logs/s...,xquad_es.json,xquad_es,f1,0.194621,gemma-2b-it,oasst2_tulu-v3,model.norm,gemma-2b-it-KL-8.0-iter1
27107,/home/fre.gilad/source/silent_direction/logs/s...,xquad_ru.json,xquad_ru,exact_match,0.015126,gemma-2b-it,oasst2_tulu-v3,model.norm,gemma-2b-it-KL-8.0-iter1
27108,/home/fre.gilad/source/silent_direction/logs/s...,xquad_ru.json,xquad_ru,f1,0.169717,gemma-2b-it,oasst2_tulu-v3,model.norm,gemma-2b-it-KL-8.0-iter1
27109,/home/fre.gilad/source/silent_direction/logs/s...,xquad_zh.json,xquad_zh,exact_match,0.001681,gemma-2b-it,oasst2_tulu-v3,model.norm,gemma-2b-it-KL-8.0-iter1


In [6]:
def kl_val(row: pd.Series) -> float:
    # Llama-2-7b-chat-hf-KL-0.0-iter1
    exp_name = row["exp_name"]
    kl_str = exp_name.split("KL-")[-1].split("-")[0]
    return float(kl_str)

logprob_results["kl"] = logprob_results.apply(kl_val, axis=1)
logprob_norm_results["kl"] = logprob_norm_results.apply(kl_val, axis=1)
generative_results["kl"] = generative_results.apply(kl_val, axis=1)

raw_results["kl"] = raw_results.apply(kl_val, axis=1)

In [7]:
def run_id(df: pd.DataFrame, id_cols:list[str]) -> pd.Series:
    return df[id_cols].astype(str).agg("-".join, axis=1)

run_id_cols = ["model_name", "layer_name", "exp_name"]

logprob_results["run_id"] = run_id(logprob_results, run_id_cols)
logprob_norm_results["run_id"] = run_id(logprob_norm_results, run_id_cols)
generative_results["run_id"] = run_id(generative_results, run_id_cols)

raw_results["run_id"] = run_id(raw_results, run_id_cols)

Process Raw Results

In [8]:
raw_results['benchmark_metric'] = raw_results['benchmark'] + "/" + raw_results['metric'] 

In [9]:
raw_results.columns.tolist()
pivot_raw_results = raw_results.pivot_table(index=["model_name", "layer_name", 'kl'], columns="benchmark_metric", values="value")


In [10]:
spoadkfnakosjdlf = 15 + 50
pivot_raw_results['global/kl_div'] = (15/spoadkfnakosjdlf) * pivot_raw_results['eval-oasst2/kl_div'] + (50/spoadkfnakosjdlf) * pivot_raw_results['eval-tulu-v3/kl_div']
pivot_raw_results['global/proj_l2_rel'] = (15/spoadkfnakosjdlf) * pivot_raw_results['eval-oasst2/proj_l2_rel'] + (50/spoadkfnakosjdlf) * pivot_raw_results['eval-tulu-v3/proj_l2_rel']


In [11]:
# Get the reference values where kl == 0
ref_values = pivot_raw_results.xs(0.0, level='kl')[['global/kl_div', 'global/proj_l2_rel']]

# Join reference values to the pivot table
# We don't need to manually initialize the _max columns as the join will create them
if 'global/kl_div_max' in pivot_raw_results.columns:
    pivot_raw_results = pivot_raw_results.drop(columns=['global/kl_div_max', 'global/proj_l2_rel_max'])

pivot_raw_results = pivot_raw_results.join(ref_values, on=['model_name', 'layer_name'], rsuffix='_max')

pivot_raw_results = pivot_raw_results[['global/kl_div', 'global/kl_div_max', 'global/proj_l2_rel', 'global/proj_l2_rel_max']]

Concat logprob with logprob results

In [12]:
logprob_results['metric'] = "acc"
logprob_norm_results['metric'] = "acc_norm"
generative_results['metric'] = "generative"

In [13]:
# add logprob_norm_results to logprob_results
logprob_results = pd.concat([logprob_results, logprob_norm_results], ignore_index=True)
logprob_results

,benchmark,model_name,layer_name,exp_name,clean_mean,dirty_mean,clean_std,dirty_std,two_sided_tail,lower_tail,upper_tail,count,diff_prob,choice_metric,kl,run_id,metric
0,anli_r1,Llama-2-7b-chat-hf,model.embed_tokens,Llama-2-7b-chat-hf-KL-0.0-iter1,0.401416,0.350160,0.008166,0.011055,2.220916e-04,1.139196e-04,1.081719e-04,1000,0.000525,7.475083e-04,0.0,Llama-2-7b-chat-hf-model.embed_tokens-Llama-2-...,acc
1,anli_r2,Llama-2-7b-chat-hf,model.embed_tokens,Llama-2-7b-chat-hf-KL-0.0-iter1,0.391547,0.350911,0.008316,0.010933,3.145660e-03,1.741479e-03,1.404181e-03,1000,0.009762,1.290762e-02,0.0,Llama-2-7b-chat-hf-model.embed_tokens-Llama-2-...,acc
2,anli_r3,Llama-2-7b-chat-hf,model.embed_tokens,Llama-2-7b-chat-hf-KL-0.0-iter1,0.371106,0.361384,0.007587,0.011175,2.919351e-01,1.459817e-01,1.459534e-01,1200,0.663413,9.553478e-01,0.0,Llama-2-7b-chat-hf-model.embed_tokens-Llama-2-...,acc
3,mastermind_24_easy,Llama-2-7b-chat-hf,model.embed_tokens,Llama-2-7b-chat-hf-KL-0.0-iter1,0.327836,0.246694,0.008346,0.007161,0.000000e+00,0.000000e+00,0.000000e+00,1522,0.000000,0.000000e+00,0.0,Llama-2-7b-chat-hf-model.embed_tokens-Llama-2-...,acc
4,mastermind_35_easy,Llama-2-7b-chat-hf,model.embed_tokens,Llama-2-7b-chat-hf-KL-0.0-iter1,0.469538,0.220347,0.006696,0.005856,2.220446e-16,6.592567e-172,2.220446e-16,1856,0.000000,2.220446e-16,0.0,Llama-2-7b-chat-hf-model.embed_tokens-Llama-2-...,acc
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6829,blimp,gemma-2b-it,model.norm,gemma-2b-it-KL-0.5-iter1,0.524923,0.525487,0.001917,0.002711,3.862504e-01,1.929640e-01,1.932864e-01,33500,1.000000,1.386250e+00,0.5,gemma-2b-it-model.norm-gemma-2b-it-KL-0.5-iter1,acc_norm
6830,blimp,gemma-2b-it,model.norm,gemma-2b-it-KL-1.0-iter1,0.524923,0.525255,0.001917,0.002711,3.904710e-01,1.950898e-01,1.953811e-01,33500,1.000000,1.390471e+00,1.0,gemma-2b-it-model.norm-gemma-2b-it-KL-1.0-iter1,acc_norm
6831,blimp,gemma-2b-it,model.norm,gemma-2b-it-KL-2.0-iter1,0.524923,0.525545,0.001917,0.002710,3.848854e-01,1.922778e-01,1.926076e-01,33500,1.000000,1.384885e+00,2.0,gemma-2b-it-model.norm-gemma-2b-it-KL-2.0-iter1,acc_norm
6832,blimp,gemma-2b-it,model.norm,gemma-2b-it-KL-4.0-iter1,0.524923,0.528749,0.001917,0.002706,1.837224e-01,9.166953e-02,9.205285e-02,33500,0.999982,1.183704e+00,4.0,gemma-2b-it-model.norm-gemma-2b-it-KL-4.0-iter1,acc_norm


In [14]:
merge_keys = ["model_name", "layer_name", "kl"]

logprob_results = logprob_results.merge(
    pivot_raw_results.reset_index(),
    on=merge_keys,
    how="left",
)

generative_results = generative_results.merge(
    pivot_raw_results.reset_index(),
    on=merge_keys,
    how="left",
)

In [15]:
def abs_diff(df: pd.DataFrame, col1: str, col2: str, new_col: str="abs_diff") -> pd.DataFrame:
    df[new_col] = (df[col1] - df[col2]).abs()
    return df

# add absolute difference between dirty_mean and clean_mean
logprob_results = abs_diff(logprob_results, "dirty_mean", "clean_mean", "abs_diff")
logprob_norm_results = abs_diff(logprob_norm_results, "dirty_mean", "clean_mean", "abs_diff")
generative_results = abs_diff(generative_results, "value", "clean_mean", "abs_diff")

In [16]:
threshold = 0.015
generative_results['diff_prob'] = (((generative_results['value'] - generative_results['clean_mean']).abs()) < threshold).astype(float)

In [17]:
choice_metric_col = "silence_score"

In [18]:
logprob_results[choice_metric_col] = logprob_results['two_sided_tail'] + logprob_results['diff_prob']
generative_results[choice_metric_col] = generative_results['two_sided_tail'] + generative_results['diff_prob']

In [19]:
logprob_results['diff_prob'].describe()

count    6834.000000
mean        0.620582
std         0.316642
min         0.000000
25%         0.335946
50%         0.722312
75%         0.888569
max         1.000000
Name: diff_prob, dtype: float64

In [20]:
# mask = generative_results['abs_diff'] < 0.015
# a=generative_results[mask].groupby(['model_name', 'layer_name', 'kl'])['two_sided_tail'].describe().reset_index()

## Now AGREGRATE

In [21]:
def agg_results(
    df: pd.DataFrame,
    prob_col: str = "two_sided_tail",
    new_col: str = "two_sided_tail_agg",
    group_by_cols: list[str] | None = None,
    keep_cols: list[str] | None = None,
) -> pd.DataFrame:
    keep_cols_local = list(keep_cols) if keep_cols is not None else []
    keep_cols_local = list(dict.fromkeys([*keep_cols_local, prob_col]))  # unique, order-preserving

    group_cols = list(group_by_cols) if group_by_cols is not None else []
    selected_cols = list(dict.fromkeys([*group_cols, *keep_cols_local]))

    if group_by_cols is not None:
        idx = df.groupby(group_by_cols, sort=False)[prob_col].idxmin()
        agg_df = df.loc[idx, selected_cols].reset_index(drop=True)
    else:
        idx = df[prob_col].idxmin()
        agg_df = df.loc[[idx], selected_cols].reset_index(drop=True)

    agg_df = agg_df.rename(columns={prob_col: new_col})
    return agg_df

In [22]:
raw_keep_cols = ['global/kl_div', 'global/kl_div_max', 'global/proj_l2_rel', 'global/proj_l2_rel_max', 'diff_prob', 'two_sided_tail', 'lower_tail', 'upper_tail', 'silence_score', "benchmark"]
agg_choice_metric_col = f"agg_{choice_metric_col}"

In [23]:
abs_diff_threshold = - 1e-10

agg_logprobs =  agg_results(
    logprob_results.copy(),
    # threshold=abs_diff_threshold,
    # diff_col="abs_diff",
    prob_col=choice_metric_col,
    new_col=agg_choice_metric_col,
    group_by_cols=["model_name", "layer_name", "kl"],
    keep_cols=raw_keep_cols,
)

agg_generative =  agg_results(
    generative_results.copy(),
    # threshold=abs_diff_threshold,
    # diff_col="abs_diff",
    prob_col=choice_metric_col,
    new_col=agg_choice_metric_col,
    group_by_cols=["model_name", "layer_name", "kl"],
    keep_cols=raw_keep_cols,
)

In [24]:
agg_generative.sort_values("agg_silence_score", ascending=False)

,model_name,layer_name,kl,global/kl_div,global/kl_div_max,global/proj_l2_rel,global/proj_l2_rel_max,diff_prob,two_sided_tail,lower_tail,upper_tail,agg_silence_score,benchmark
151,Qwen2.5-3B-Instruct,model.norm,0.125,0.068199,1.690635,0.319555,0.328088,0.0,0.392169,0.196085,0.196085,0.392169,ifeval
147,Qwen2.5-3B-Instruct,model.layers.24,2.000,0.010173,2.623426,0.000387,0.372008,0.0,0.335589,0.167795,0.167795,0.335589,metabench_gsm8k
111,Qwen2.5-14B-Instruct,model.embed_tokens,0.250,0.016254,0.134756,0.052910,0.052672,0.0,0.274411,0.137205,0.137205,0.274411,ifeval
5,Llama-2-7b-chat-hf,model.embed_tokens,2.000,0.001979,3.044495,0.035814,0.051514,0.0,0.261276,0.130638,0.130638,0.261276,mbpp
155,Qwen2.5-3B-Instruct,model.norm,2.000,0.016237,1.690635,0.294276,0.328088,0.0,0.230081,0.115041,0.115041,0.230081,metabench_gsm8k
...,...,...,...,...,...,...,...,...,...,...,...,...,...
101,Qwen2.5-0.5B-Instruct,model.layers.8,2.000,0.047002,2.691051,0.002837,0.257947,0.0,0.000000,0.000000,0.000000,0.000000,jsonschema_bench_medium
100,Qwen2.5-0.5B-Instruct,model.layers.8,1.000,0.040489,2.691051,0.171038,0.257947,0.0,0.000000,0.000000,0.000000,0.000000,jsonschema_bench_medium
99,Qwen2.5-0.5B-Instruct,model.layers.8,0.250,0.105772,2.691051,0.201218,0.257947,0.0,0.000000,0.000000,0.000000,0.000000,jsonschema_bench_medium
81,Qwen2.5-0.5B-Instruct,model.embed_tokens,0.250,0.056160,0.397620,0.171161,0.179052,0.0,0.000000,0.000000,0.000000,0.000000,jsonschema_bench_medium


In [25]:
agg_logprobs['choice_metric'] = agg_logprobs['global/proj_l2_rel'] * agg_logprobs[agg_choice_metric_col]
agg_generative['choice_metric'] = agg_generative['global/proj_l2_rel'] * agg_generative[agg_choice_metric_col]

In [26]:
metrict_to_keep_part_2 = ["diff_prob", "two_sided_tail", "lower_tail", "upper_tail", "agg_silence_score", "benchmark"]
# create ne dataframe with model_name, layer_name, kl, choice_metric for both logprobs and generative
agg_logprobs_choice = agg_logprobs[["model_name", "layer_name", "kl", 'choice_metric']+metrict_to_keep_part_2]
agg_generative_choice = agg_generative[["model_name", "layer_name", "kl", 'choice_metric']+metrict_to_keep_part_2]

agg_choice = agg_logprobs_choice.merge(
    agg_generative_choice,
    on=["model_name", "layer_name", "kl"],
    suffixes=("_logprob", "_generative"),
)

agg_logprobs_metrics = agg_logprobs[["model_name", "layer_name", "kl", "global/proj_l2_rel", "global/proj_l2_rel_max", "global/kl_div", "global/kl_div_max"]]

agg_choice = agg_choice.merge(
    agg_logprobs_metrics,
    on=["model_name", "layer_name", "kl"],
    how="left",
)

agg_choice

,model_name,layer_name,kl,choice_metric_logprob,diff_prob_logprob,two_sided_tail_logprob,lower_tail_logprob,upper_tail_logprob,agg_silence_score_logprob,benchmark_logprob,...,diff_prob_generative,two_sided_tail_generative,lower_tail_generative,upper_tail_generative,agg_silence_score_generative,benchmark_generative,global/proj_l2_rel,global/proj_l2_rel_max,global/kl_div,global/kl_div_max
0,Llama-2-7b-chat-hf,model.embed_tokens,0.000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,mastermind_24_easy,...,0.0,0.000034,0.000017,0.000017,0.000034,xquad_es,0.051514,0.051514,3.044495,3.044495
1,Llama-2-7b-chat-hf,model.embed_tokens,0.125,0.002826,0.035947,0.027814,0.015032,0.012782,0.063760,toxigen,...,0.0,0.019804,0.009902,0.009902,0.019804,ifeval,0.044323,0.051514,0.022527,3.044495
2,Llama-2-7b-chat-hf,model.embed_tokens,0.250,0.011990,0.170436,0.105620,0.056145,0.049476,0.276057,toxigen,...,0.0,0.072827,0.036414,0.036414,0.072827,ifeval,0.043432,0.051514,0.012225,3.044495
3,Llama-2-7b-chat-hf,model.embed_tokens,0.500,0.032839,0.271115,0.524421,0.269170,0.255251,0.795536,metabench_winogrande,...,0.0,0.018180,0.009090,0.009090,0.018180,jsonschema_bench_easy,0.041279,0.051514,0.009540,3.044495
4,Llama-2-7b-chat-hf,model.embed_tokens,1.000,0.030190,0.271112,0.524427,0.269168,0.255259,0.795539,metabench_winogrande,...,0.0,0.010900,0.005450,0.005450,0.010900,ifeval,0.037949,0.051514,0.004707,3.044495
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
193,gemma-2b-it,model.norm,0.500,0.192346,0.271146,0.404886,0.208708,0.196178,0.676032,metabench_winogrande,...,0.0,0.024215,0.012108,0.012108,0.024215,jsonschema_bench_easy,0.284523,0.306153,0.024408,4.467519
194,gemma-2b-it,model.norm,1.000,0.178630,0.271146,0.404889,0.208707,0.196182,0.676035,metabench_winogrande,...,0.0,0.001797,0.000899,0.000899,0.001797,xquad_en,0.264232,0.306153,0.023484,4.467519
195,gemma-2b-it,model.norm,2.000,0.172925,0.271146,0.404888,0.208708,0.196180,0.676034,metabench_winogrande,...,0.0,0.044467,0.022233,0.022233,0.044467,jsonschema_bench_medium,0.255794,0.306153,0.026507,4.467519
196,gemma-2b-it,model.norm,4.000,0.030179,0.144347,0.013418,0.007101,0.006317,0.157765,toxigen,...,0.0,0.014607,0.007303,0.007303,0.014607,jsonschema_bench_medium,0.191288,0.306153,0.312896,4.467519


In [27]:
# add agg_choice_metric column that is min of choice_metric_logprob and choice_metric_generative
agg_choice["agg_choice_metric"] = agg_choice[[f'choice_metric_logprob', f'choice_metric_generative']].min(axis=1)

agg_choice['agg_silent_metric'] = agg_choice['agg_choice_metric'] / agg_choice['global/proj_l2_rel']
# agg_choice['silent_metric_logprob'] = agg_choice['choice_metric_logprob'] / agg_choice['global/proj_l2_rel']
# agg_choice['silent_metric_generative'] = agg_choice['choice_metric_generative'] / agg_choice['global/proj_l2_rel']


agg_choice
# for each group of (model_name, layer_name) take the kl value with the highest value 

,model_name,layer_name,kl,choice_metric_logprob,diff_prob_logprob,two_sided_tail_logprob,lower_tail_logprob,upper_tail_logprob,agg_silence_score_logprob,benchmark_logprob,...,lower_tail_generative,upper_tail_generative,agg_silence_score_generative,benchmark_generative,global/proj_l2_rel,global/proj_l2_rel_max,global/kl_div,global/kl_div_max,agg_choice_metric,agg_silent_metric
0,Llama-2-7b-chat-hf,model.embed_tokens,0.000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,mastermind_24_easy,...,0.000017,0.000017,0.000034,xquad_es,0.051514,0.051514,3.044495,3.044495,0.000000,0.000000
1,Llama-2-7b-chat-hf,model.embed_tokens,0.125,0.002826,0.035947,0.027814,0.015032,0.012782,0.063760,toxigen,...,0.009902,0.009902,0.019804,ifeval,0.044323,0.051514,0.022527,3.044495,0.000878,0.019804
2,Llama-2-7b-chat-hf,model.embed_tokens,0.250,0.011990,0.170436,0.105620,0.056145,0.049476,0.276057,toxigen,...,0.036414,0.036414,0.072827,ifeval,0.043432,0.051514,0.012225,3.044495,0.003163,0.072827
3,Llama-2-7b-chat-hf,model.embed_tokens,0.500,0.032839,0.271115,0.524421,0.269170,0.255251,0.795536,metabench_winogrande,...,0.009090,0.009090,0.018180,jsonschema_bench_easy,0.041279,0.051514,0.009540,3.044495,0.000750,0.018180
4,Llama-2-7b-chat-hf,model.embed_tokens,1.000,0.030190,0.271112,0.524427,0.269168,0.255259,0.795539,metabench_winogrande,...,0.005450,0.005450,0.010900,ifeval,0.037949,0.051514,0.004707,3.044495,0.000414,0.010900
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
193,gemma-2b-it,model.norm,0.500,0.192346,0.271146,0.404886,0.208708,0.196178,0.676032,metabench_winogrande,...,0.012108,0.012108,0.024215,jsonschema_bench_easy,0.284523,0.306153,0.024408,4.467519,0.006890,0.024215
194,gemma-2b-it,model.norm,1.000,0.178630,0.271146,0.404889,0.208707,0.196182,0.676035,metabench_winogrande,...,0.000899,0.000899,0.001797,xquad_en,0.264232,0.306153,0.023484,4.467519,0.000475,0.001797
195,gemma-2b-it,model.norm,2.000,0.172925,0.271146,0.404888,0.208708,0.196180,0.676034,metabench_winogrande,...,0.022233,0.022233,0.044467,jsonschema_bench_medium,0.255794,0.306153,0.026507,4.467519,0.011374,0.044467
196,gemma-2b-it,model.norm,4.000,0.030179,0.144347,0.013418,0.007101,0.006317,0.157765,toxigen,...,0.007303,0.007303,0.014607,jsonschema_bench_medium,0.191288,0.306153,0.312896,4.467519,0.002794,0.014607


In [28]:
# Cleaner: break ties with kl first, then pick max agg_choice_metric per group via idxmax.
_tmp = agg_choice.sort_values(["model_name", "layer_name", "kl"], ascending=[True, True, False])
_idx = _tmp.groupby(["model_name", "layer_name"])['agg_choice_metric'].idxmax()

best_kl_df = (
    _tmp.loc[_idx, agg_choice.columns.tolist()]
    .sort_values(["model_name", "layer_name"])
).reset_index(drop=True)

best_kl_df

,model_name,layer_name,kl,choice_metric_logprob,diff_prob_logprob,two_sided_tail_logprob,lower_tail_logprob,upper_tail_logprob,agg_silence_score_logprob,benchmark_logprob,...,lower_tail_generative,upper_tail_generative,agg_silence_score_generative,benchmark_generative,global/proj_l2_rel,global/proj_l2_rel_max,global/kl_div,global/kl_div_max,agg_choice_metric,agg_silent_metric
0,Llama-2-7b-chat-hf,model.embed_tokens,2.000,0.028491,0.271111,5.244287e-01,2.691672e-01,2.552615e-01,0.795540,metabench_winogrande,...,0.130638,0.130638,0.261276,mbpp,0.035814,0.051514,0.001979,3.044495,0.009357,0.261276
1,Llama-2-7b-chat-hf,model.layers.0,0.500,0.050195,0.039992,8.623066e-02,4.502372e-02,4.120694e-02,0.126222,metabench_hellaswag,...,0.016029,0.016029,0.032057,mbpp,0.397674,0.454612,0.062223,6.306403,0.012748,0.032057
2,Llama-2-7b-chat-hf,model.layers.11,2.000,0.005848,0.106260,7.460014e-02,3.729512e-02,3.730503e-02,0.180860,toxigen,...,0.106602,0.106602,0.213204,metabench_gsm8k,0.032334,0.090074,0.004169,1.407147,0.005848,0.180860
3,Llama-2-7b-chat-hf,model.layers.21,4.000,0.044970,0.271106,5.244379e-01,2.691641e-01,2.552738e-01,0.795544,metabench_winogrande,...,0.111111,0.111111,0.222222,ifeval,0.056527,0.097992,0.003810,0.795353,0.012562,0.222222
4,Llama-2-7b-chat-hf,model.norm,0.125,0.076042,0.271106,5.244379e-01,2.691641e-01,2.552738e-01,0.795544,metabench_winogrande,...,0.064806,0.064806,0.129612,ifeval,0.095584,0.101190,0.017880,0.632451,0.012389,0.129612
5,Phi-3-mini-4k-instruct,model.embed_tokens,1.000,0.033659,0.271084,5.244443e-01,2.682156e-01,2.562287e-01,0.795529,metabench_winogrande,...,0.009453,0.009453,0.018906,metabench_gsm8k,0.042310,0.045035,0.004852,0.197981,0.000800,0.018906
6,Phi-3-mini-4k-instruct,model.layers.0,2.000,0.010809,0.008826,1.242157e-02,6.585171e-03,5.836401e-03,0.021248,anli_r1,...,0.011280,0.011280,0.022559,xquad_ar,0.508712,0.596129,0.043053,1.521296,0.010809,0.021248
7,Phi-3-mini-4k-instruct,model.layers.11,2.000,0.055194,0.087013,6.835747e-02,3.562272e-02,3.273475e-02,0.155371,anli_r1,...,0.003170,0.003170,0.006339,ifeval,0.355240,0.501323,0.033106,1.971606,0.002252,0.006339
8,Phi-3-mini-4k-instruct,model.layers.21,1.000,0.220277,0.192075,3.230746e-01,1.620867e-01,1.609879e-01,0.515150,metabench_hellaswag,...,0.001455,0.001455,0.002909,ifeval,0.427598,0.492989,0.064960,1.699576,0.001244,0.002909
9,Phi-3-mini-4k-instruct,model.norm,0.500,0.308146,0.461798,1.515458e-01,7.677741e-02,7.476834e-02,0.613344,mastermind_46_easy,...,0.000526,0.000526,0.001053,xquad_ru,0.502403,0.516003,0.014199,1.674800,0.000529,0.001053


In [35]:
rename_map = {
    # Experiment
    'model_name': 'Model',
    'layer_name': 'Layer',
    'kl': r'$\lambda$',
    # Experiment Results
    'agg_silent_metric': r'$\mathrm{Sil}$',
    'global/proj_l2_rel': r'$E_{L2}$',
    'global/proj_l2_rel_max': r'$\max E_{L2}$',
    'global/kl_div': r'$\mathcal{L}_{KL}$',
    'global/kl_div_max': r'$\max \mathcal{L}_{KL}$',
    # Logprob Results
    'diff_prob_logprob': r'$\mathrm{Diff}_{\text{logprob}}$',
    'two_sided_tail_logprob': r'$\mathrm{Tail}_{\text{logprob}}$',
    'agg_silence_score_logprob': r'$\mathrm{Sil}_{\text{logprob}}$',
    # Generative Results
    'diff_prob_generative': r'$\mathrm{Diff}_{\text{generative}}$',
    'two_sided_tail_generative': r'$\mathrm{Tail}_{\text{generative}}$',
    'agg_silence_score_generative': r'$\mathrm{Sil}_{\text{generative}}$',
    "benchmark_generative": "benchmark_generative",
    "benchmark_logprob": "benchmark_logprob",
}
mask_cols = [k for k in rename_map.keys()]
best_kl_df_renamed = best_kl_df[mask_cols].rename(columns=rename_map)

In [36]:
best_kl_df_renamed.sort_values(rename_map['agg_silent_metric'], ascending=False)

,Model,Layer,$\lambda$,$\mathrm{Sil}$,$E_{L2}$,$\max E_{L2}$,$\mathcal{L}_{KL}$,$\max \mathcal{L}_{KL}$,$\mathrm{Diff}_{\text{logprob}}$,$\mathrm{Tail}_{\text{logprob}}$,$\mathrm{Sil}_{\text{logprob}}$,$\mathrm{Diff}_{\text{generative}}$,$\mathrm{Tail}_{\text{generative}}$,$\mathrm{Sil}_{\text{generative}}$,benchmark_generative,benchmark_logprob
21,Qwen2.5-3B-Instruct,model.norm,0.125,0.392169,0.319555,0.328088,0.068199,1.690635,0.271133,5.244371e-01,0.795570,0.0,0.392169,0.392169,ifeval,metabench_winogrande
20,Qwen2.5-3B-Instruct,model.layers.24,2.000,0.335589,0.000387,0.372008,0.010173,2.623426,0.271131,5.244452e-01,0.795576,0.0,0.335589,0.335589,metabench_gsm8k,metabench_winogrande
0,Llama-2-7b-chat-hf,model.embed_tokens,2.000,0.261276,0.035814,0.051514,0.001979,3.044495,0.271111,5.244287e-01,0.795540,0.0,0.261276,0.261276,mbpp,metabench_winogrande
3,Llama-2-7b-chat-hf,model.layers.21,4.000,0.222222,0.056527,0.097992,0.003810,0.795353,0.271106,5.244379e-01,0.795544,0.0,0.222222,0.222222,ifeval,metabench_winogrande
24,gemma-2b-it,model.layers.12,0.500,0.188497,0.123231,0.167434,0.035517,5.093478,0.537105,7.022424e-02,0.607329,0.0,0.188497,0.188497,jsonschema_bench_medium,mastermind_24_easy
2,Llama-2-7b-chat-hf,model.layers.11,2.000,0.180860,0.032334,0.090074,0.004169,1.407147,0.106260,7.460014e-02,0.180860,0.0,0.213204,0.213204,metabench_gsm8k,toxigen
4,Llama-2-7b-chat-hf,model.norm,0.125,0.129612,0.095584,0.101190,0.017880,0.632451,0.271106,5.244379e-01,0.795544,0.0,0.129612,0.129612,ifeval,metabench_winogrande
16,Qwen2.5-14B-Instruct,model.norm,3.000,0.118083,0.134103,0.211883,0.010046,1.228669,0.266548,5.249573e-01,0.791506,0.0,0.118083,0.118083,jsonschema_bench_medium,metabench_hellaswag
23,gemma-2b-it,model.layers.0,8.000,0.113156,0.391647,0.536371,0.013488,2.035118,0.178110,2.083527e-02,0.198945,0.0,0.113156,0.113156,jsonschema_bench_easy,toxigen
18,Qwen2.5-3B-Instruct,model.layers.0,8.000,0.072827,0.001878,0.200465,0.001814,0.300584,0.271131,5.244441e-01,0.795576,0.0,0.072827,0.072827,jsonschema_bench_hard,metabench_winogrande


In [37]:
best_kl_df_renamed

,Model,Layer,$\lambda$,$\mathrm{Sil}$,$E_{L2}$,$\max E_{L2}$,$\mathcal{L}_{KL}$,$\max \mathcal{L}_{KL}$,$\mathrm{Diff}_{\text{logprob}}$,$\mathrm{Tail}_{\text{logprob}}$,$\mathrm{Sil}_{\text{logprob}}$,$\mathrm{Diff}_{\text{generative}}$,$\mathrm{Tail}_{\text{generative}}$,$\mathrm{Sil}_{\text{generative}}$,benchmark_generative,benchmark_logprob
0,Llama-2-7b-chat-hf,model.embed_tokens,2.000,0.261276,0.035814,0.051514,0.001979,3.044495,0.271111,5.244287e-01,0.795540,0.0,0.261276,0.261276,mbpp,metabench_winogrande
1,Llama-2-7b-chat-hf,model.layers.0,0.500,0.032057,0.397674,0.454612,0.062223,6.306403,0.039992,8.623066e-02,0.126222,0.0,0.032057,0.032057,mbpp,metabench_hellaswag
2,Llama-2-7b-chat-hf,model.layers.11,2.000,0.180860,0.032334,0.090074,0.004169,1.407147,0.106260,7.460014e-02,0.180860,0.0,0.213204,0.213204,metabench_gsm8k,toxigen
3,Llama-2-7b-chat-hf,model.layers.21,4.000,0.222222,0.056527,0.097992,0.003810,0.795353,0.271106,5.244379e-01,0.795544,0.0,0.222222,0.222222,ifeval,metabench_winogrande
4,Llama-2-7b-chat-hf,model.norm,0.125,0.129612,0.095584,0.101190,0.017880,0.632451,0.271106,5.244379e-01,0.795544,0.0,0.129612,0.129612,ifeval,metabench_winogrande
5,Phi-3-mini-4k-instruct,model.embed_tokens,1.000,0.018906,0.042310,0.045035,0.004852,0.197981,0.271084,5.244443e-01,0.795529,0.0,0.018906,0.018906,metabench_gsm8k,metabench_winogrande
6,Phi-3-mini-4k-instruct,model.layers.0,2.000,0.021248,0.508712,0.596129,0.043053,1.521296,0.008826,1.242157e-02,0.021248,0.0,0.022559,0.022559,xquad_ar,anli_r1
7,Phi-3-mini-4k-instruct,model.layers.11,2.000,0.006339,0.355240,0.501323,0.033106,1.971606,0.087013,6.835747e-02,0.155371,0.0,0.006339,0.006339,ifeval,anli_r1
8,Phi-3-mini-4k-instruct,model.layers.21,1.000,0.002909,0.427598,0.492989,0.064960,1.699576,0.192075,3.230746e-01,0.515150,0.0,0.002909,0.002909,ifeval,metabench_hellaswag
9,Phi-3-mini-4k-instruct,model.norm,0.500,0.001053,0.502403,0.516003,0.014199,1.674800,0.461798,1.515458e-01,0.613344,0.0,0.001053,0.001053,xquad_ru,mastermind_46_easy


In [38]:
def fmt(x, k=4):
    return f"{x:.{k}f}".rstrip("0").rstrip(".")

print(
    best_kl_df_renamed.to_latex(
        index=False,
        float_format=lambda x: fmt(x, k=4)
    )
)

\begin{tabular}{llrrrrrrrrrrrrll}
\toprule
Model & Layer & $\lambda$ & $\mathrm{Sil}$ & $E_{L2}$ & $\max E_{L2}$ & $\mathcal{L}_{KL}$ & $\max \mathcal{L}_{KL}$ & $\mathrm{Diff}_{\text{logprob}}$ & $\mathrm{Tail}_{\text{logprob}}$ & $\mathrm{Sil}_{\text{logprob}}$ & $\mathrm{Diff}_{\text{generative}}$ & $\mathrm{Tail}_{\text{generative}}$ & $\mathrm{Sil}_{\text{generative}}$ & benchmark_generative & benchmark_logprob \\
\midrule
Llama-2-7b-chat-hf & model.embed_tokens & 2 & 0.2613 & 0.0358 & 0.0515 & 0.002 & 3.0445 & 0.2711 & 0.5244 & 0.7955 & 0 & 0.2613 & 0.2613 & mbpp & metabench_winogrande \\
Llama-2-7b-chat-hf & model.layers.0 & 0.5 & 0.0321 & 0.3977 & 0.4546 & 0.0622 & 6.3064 & 0.04 & 0.0862 & 0.1262 & 0 & 0.0321 & 0.0321 & mbpp & metabench_hellaswag \\
Llama-2-7b-chat-hf & model.layers.11 & 2 & 0.1809 & 0.0323 & 0.0901 & 0.0042 & 1.4071 & 0.1063 & 0.0746 & 0.1809 & 0 & 0.2132 & 0.2132 & metabench_gsm8k & toxigen \\
Llama-2-7b-chat-hf & model.layers.21 & 4 & 0.2222 & 0.0565 & 0.098

In [39]:
def layer_order(layer_name: str) -> int:
    if ".layers." in layer_name:
        return int(layer_name.split(".")[-1])
    elif "embed_tokens" in layer_name:
        return -1
    else:
        return 1000

In [40]:
def visualize(
        df:pd.DataFrame,
        res_name: str,
        abs_diff_threshold: float,
        row:str = None,
    ):
    g = sns.catplot(
        data=df,
        x="two_sided_tail_agg",
        y="layer_name",
        kind="bar",
        col="model_name",
        row=row,
        order=sorted(df["layer_name"].unique(), key=layer_order),
    )
    g.figure.suptitle(f"Aggregated {res_name} two-sided tail values (threshold={abs_diff_threshold})", y=1.02)